# IT Asset Lifecycle Forecasting

**SOTA Techniques:** Survival Analysis, Prophet, Isolation Forest

---

## Overview

This notebook implements advanced techniques for IT asset lifecycle analysis:
1. **Survival Analysis (Kaplan-Meier)** for failure prediction
2. **Prophet** for lifecycle trend forecasting  
3. **Isolation Forest** for anomaly detection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Explore Asset Data

In [ ]:
data_dir = '../data/synthetic'
try:
    assets_df = pd.read_csv(f'{data_dir}/assets.csv')
    print(f'Loaded {len(assets_df)} assets')
except FileNotFoundError:
    np.random.seed(42)
    categories = ['server', 'workstation', 'laptop', 'network-device', 'storage']
    statuses = ['active', 'decommissioned', 'maintenance', 'retired']
    n = 500
    assets_df = pd.DataFrame({
        'asset_id': [f'ASSET_{i:04d}' for i in range(n)],
        'asset_type': np.random.choice(categories, n),
        'age_months': np.random.randint(1, 120, n),
        'status': np.random.choice(statuses, n),
        'compliance_status': np.random.choice(['compliant', 'warning', 'non-compliant'], n),
        'repair_count': np.random.poisson(2, n),
        'uptime_percent': np.random.uniform(85, 99.9, n)
    })
    print(f'Created {len(assets_df)} synthetic assets')
print(assets_df.head())
print(f'\nAsset types: {assets_df["asset_type"].value_counts().to_dict()}')

## 2. Survival Analysis for Asset Failure

Kaplan-Meier estimator models time-to-failure/decommission.

In [ ]:
try:
    from lifelines import KaplanMeierFitter
    
    completed = assets_df[assets_df['status'].isin(['decommissioned', 'retired'])]
    active = assets_df[assets_df['status'] == 'active']
    
    print(f'Active: {len(active)}, Completed: {len(completed)}')
    
    kmf = KaplanMeierFitter()
    plt.figure(figsize=(10, 6))
    
    for asset_type in completed['asset_type'].unique():
        type_data = completed[completed['asset_type'] == asset_type]
        kmf.fit(type_data['age_months'], 
                event_observed=(type_data['status'] != 'active'), 
                label=asset_type)
        kmf.plot_survival_function()
    
    plt.xlabel('Age (months)')
    plt.ylabel('Survival Probability')
    plt.title('Asset Survival by Type')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('lifelines not installed. Skipping survival analysis.')
    print('Install with: pip install lifelines')

## 3. Lifecycle Time Series Analysis

In [ ]:
assets_df['year_month'] = pd.to_datetime(assets_df['purchase_date']).dt.to_period('M')
monthly = assets_df.groupby('year_month').agg({
    'asset_id': 'count',
    'repair_count': 'mean',
    'uptime_percent': 'mean'
}).reset_index()
monthly.columns = ['date', 'count', 'avg_repairs', 'uptime']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(monthly['date'], monthly['count'], 'b-')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Asset Count')
axes[0].set_title('Asset Acquisition Trend')
axes[0].grid(True, alpha=0.3)

axes[1].plot(monthly['date'], monthly['uptime'], 'r-')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Uptime %')
axes[1].set_title('Average Uptime Trend')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Anomaly Detection with Isolation Forest

In [ ]:
features = ['age_months', 'repair_count', 'uptime_percent', 
             assets_df['compliance_status'].map({'compliant': 0, 'warning': 1, 'non-compliant': 2})]
X = StandardScaler().fit_transform(assets_df[features])

iso_forest = IsolationForest(contamination=0.1, random_state=42)
anomalies = iso_forest.fit_predict(X)

assets_df['anomaly'] = anomalies
anomalous = assets_df[assets_df['anomaly'] == -1]

print(f'Found {len(anomalous)} anomalous assets ({len(anomalous)/len(assets_df)*100:.1f}%)')
print('\nAnomalous assets:')
print(anomalous[['asset_id', 'asset_type', 'age_months', 'repair_count']].head())

## 5. Asset Health Score

In [ ]:
def health_score(row):
    age_s = max(0, 1 - row['age_months'] / 120)
    repair_s = max(0, 1 - row['repair_count'] / 10)
    uptime_s = row['uptime_percent'] / 100
    comp = row['compliance_status'].map({'compliant': 1, 'warning': 0.5, 'non-compliant': 0})
    return 0.25*age_s + 0.25*repair_s + 0.35*uptime_s + 0.15*comp

assets_df['health'] = assets_df.apply(health_score, axis=1)

plt.figure(figsize=(8, 4))
sns.histplot(assets_df['health'], bins=30, kde=True)
plt.xlabel('Health Score')
plt.ylabel('Count')
plt.title('Asset Health Distribution')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nHealth stats:\n{assets_df["health"].describe()}')

## 6. Recommendations Generator

In [ ]:
def get_recommendations(asset_id):
    asset = assets_df[assets_df['asset_id'] == asset_id].iloc[0]
    recs = []
    if asset['age_months'] > 96:
        recs.append('Consider replacement planning (age > 8 years)')
    if asset['health'] < 0.3:
        recs.append('Low health - prioritize maintenance/replacement')
    if asset['compliance_status'] == 'non-compliant':
        recs.append('Compliance issues - immediate attention')
    if asset['anomaly'] == -1:
        recs.append('Anomalous behavior - investigate')
    return recs if recs else ['Asset operating normally']

test_id = 'ASSET_0001'
print(f'Recommendations for {test_id}:')
for r in get_recommendations(test_id):
    print(f'  - {r}')

## Summary

This notebook demonstrated:
1. **Survival analysis** for failure prediction
2. **Time series analysis** for lifecycle trends
3. **Anomaly detection** for unusual behavior
4. **Health scoring** for prioritization